# Parte 1: Model Checking Proposicional

En esta parte implementarás las funciones de **model checking** para lógica proposicional.

## Objetivos
1. Generar todos los modelos posibles para un conjunto de átomos
2. Verificar satisfacibilidad de fórmulas
3. Verificar si una fórmula es tautología
4. Verificar si una base de conocimiento implica una consulta
5. Generar tablas de verdad

In [1]:
import sys

sys.path.insert(0, "..")

from src.logic.propositional.ast import (
    Atom,
    Not,
    And,
    Or,
    Implies,
    Iff,
    evaluate,
    get_atoms,
)
from src.logic.propositional.utils import formula_to_string, print_truth_table

## Paso 0: Familiarízate con el AST

Antes de implementar, experimenta con las clases de `propositional/ast.py`.

In [2]:
# Crear fórmulas
p = Atom("p")
q = Atom("q")

# p → q
f1 = Implies(p, q)
print(f"Fórmula: {formula_to_string(f1)}")
print(f"Átomos: {get_atoms(f1)}")
print(f"Evaluación con p=True, q=False: {evaluate(f1, {'p': True, 'q': False})}")
print(f"Evaluación con p=True, q=True: {evaluate(f1, {'p': True, 'q': True})}")

Fórmula: (p → q)
Átomos: frozenset({'p', 'q'})
Evaluación con p=True, q=False: False
Evaluación con p=True, q=True: True


In [3]:
# Fórmula más compleja: (p ∧ q) → (p ∨ r)
r = Atom("r")
f2 = Implies(And(p, q), Or(p, r))
print(f"Fórmula: {formula_to_string(f2)}")
print(f"Átomos: {get_atoms(f2)}")

Fórmula: ((p ∧ q) → (p ∨ r))
Átomos: frozenset({'p', 'r', 'q'})


### Bicondicional (↔)

El bicondicional `Iff(p, q)` es verdadero cuando ambos tienen el mismo valor de verdad: ambos verdaderos o ambos falsos. Se lee "p si y solo si q".

In [4]:
# Bicondicional: p ↔ q
f3 = Iff(p, q)
print(f"Fórmula: {formula_to_string(f3)}")
print(f"p=True,  q=True:  {evaluate(f3, {'p': True, 'q': True})}")
print(f"p=True,  q=False: {evaluate(f3, {'p': True, 'q': False})}")
print(f"p=False, q=True:  {evaluate(f3, {'p': False, 'q': True})}")
print(f"p=False, q=False: {evaluate(f3, {'p': False, 'q': False})}")

# Construye la tabla tú mismo (get_atoms + evaluate), luego imprímela
atoms = sorted(get_atoms(f3))
headers = atoms + [formula_to_string(f3)]
rows = []
n = len(atoms)
for i in range(2**n):
    model = {atom: bool((i >> (n - 1 - j)) & 1) for j, atom in enumerate(atoms)}
    rows.append([model[a] for a in atoms] + [evaluate(f3, model)])

print("\nTabla de verdad de p ↔ q:")
print_truth_table(headers, rows)

Fórmula: (p ↔ q)
p=True,  q=True:  True
p=True,  q=False: False
p=False, q=True:  False
p=False, q=False: True

Tabla de verdad de p ↔ q:
| p     | q     | (p ↔ q) |
|-------|-------|---------|
| False | False | True    |
| False | True  | False   |
| True  | False | False   |
| True  | True  | True    |


## Paso 1: Implementar `get_all_models`

Abre `src/logic/propositional/model_checking.py` e implementa la función `get_all_models`.

**Hint:** Para n átomos, hay 2^n modelos posibles. Piensa en cómo los números del 0 al 2^n - 1 en binario te dan todas las combinaciones de True/False.

In [5]:
from src.logic.propositional.model_checking import get_all_models

# Prueba tu implementación
models = get_all_models({"p", "q", "r"})
print(f"Número de modelos: {len(models)}")
for m in models:
    print(f"  {m}")

# Debe imprimir 8 modelos con todas las combinaciones

NotImplementedError: Implement get_all_models()

## Paso 2: Implementar `check_satisfiable`

Una fórmula es **satisfacible** si existe al menos un modelo donde es verdadera.

**Hint:** Usa `get_all_models` y `evaluate`.

In [ ]:
from src.logic.propositional.model_checking import check_satisfiable

# p ∧ q es satisfacible
sat, model = check_satisfiable(And(Atom('p'), Atom('q')))
print(f"p ∧ q es satisfacible: {sat}, modelo: {model}")

# p ∧ ¬p es insatisfacible
sat, model = check_satisfiable(And(Atom('p'), Not(Atom('p'))))
print(f"p ∧ ¬p es satisfacible: {sat}, modelo: {model}")

## Paso 3: Implementar `check_valid`

Una fórmula es **válida** (tautología) si es verdadera en todos los modelos.

**Hint:** φ es válida ⟺ ¬φ es insatisfacible.

In [ ]:
from src.logic.propositional.model_checking import check_valid

# p ∨ ¬p es tautología
print(f"p ∨ ¬p es válida: {check_valid(Or(Atom('p'), Not(Atom('p'))))}")

# p → p es tautología
print(f"p → p es válida: {check_valid(Implies(Atom('p'), Atom('p')))}")

# p no es tautología
print(f"p es válida: {check_valid(Atom('p'))}")

## Paso 4: Implementar `check_entailment`

KB |= q si en todo modelo donde la KB es verdadera, q también lo es.

**Hint:** KB |= q ⟺ KB ∧ ¬q es insatisfacible.

In [ ]:
from src.logic.propositional.model_checking import check_entailment

# Modus ponens: {p → q, p} |= q
kb = [Implies(Atom('p'), Atom('q')), Atom('p')]
print(f"Modus ponens: {check_entailment(kb, Atom('q'))}")

# No se puede derivar q solo de p → q
kb = [Implies(Atom('p'), Atom('q'))]
print(f"Solo implicación: {check_entailment(kb, Atom('q'))}")

## Paso 5: Implementar `truth_table`

In [ ]:
from src.logic.propositional.model_checking import truth_table

# Tabla de verdad de p → q
table = truth_table(Implies(Atom('p'), Atom('q')))
for model, result in table:
    print(f"  {model} → {result}")

### Visualizar tabla de verdad con `print_truth_table`

`print_truth_table(headers, rows)` **solo imprime** una tabla que tú ya calculaste.
No genera los 2ⁿ modelos: arma `headers` y `rows` con `get_atoms` / `evaluate`
(o con tu `truth_table`) y pásalos a la función.

In [ ]:
def build_truth_table_rows(formula):
    """Arma headers y rows a partir de truth_table (o el bucle con evaluate)."""
    atoms = sorted(get_atoms(formula))
    headers = atoms + [formula_to_string(formula)]
    table = truth_table(formula)
    rows = [[model[a] for a in atoms] + [result] for model, result in table]
    return headers, rows


# Tabla de verdad de p → q (formateada)
headers, rows = build_truth_table_rows(Implies(Atom('p'), Atom('q')))
print_truth_table(headers, rows)

print()

# Tabla con 3 átomos: (p ∧ q) → (p ∨ r)
headers, rows = build_truth_table_rows(
    Implies(And(Atom('p'), Atom('q')), Or(Atom('p'), Atom('r')))
)
print_truth_table(headers, rows)

## Aplicación: Grupo A del Mundial FIFA 2026

Ahora aplica tu model checking a un escenario proposicional inspirado en el **Grupo A** (`MEX`, `RSA`, `KOR`, `CZE`).

Resultados relevantes del CSV:
- `MEX 2-0 RSA`, `MEX 1-0 KOR`, `CZE 0-3 MEX` → México gana sus 3 partidos → **9 pts**
- `CZE` solo suma **1 punto** (empate 1-1 con RSA) y queda último

La KB codifica hechos de victoria/puntos y reglas del estilo *victoria ⇒ puntos* y *9 pts ⇒ clasifica* (top-2).

In [ ]:
# Grupo A real: MEX ganó vs RSA, KOR y CZE → 9 pts → clasifica
# CZE con 1 pt → eliminado (último del grupo)
kb_fifa = [
    # Hechos: México venció a sus 3 rivales del grupo
    Atom('MEX_vence_RSA'),
    Atom('MEX_vence_KOR'),
    Atom('MEX_vence_CZE'),
    # Tres victorias ⇒ 9 puntos
    Implies(
        And(Atom('MEX_vence_RSA'), Atom('MEX_vence_KOR'), Atom('MEX_vence_CZE')),
        Atom('MEX_9pts'),
    ),
    # 9 puntos ⇒ clasificación directa (top-2 por puntos)
    Implies(Atom('MEX_9pts'), Atom('MEX_clasifica')),
    # Hecho: Chequia terminó con 1 punto
    Atom('CZE_1pt'),
    # 1 punto ⇒ eliminado (no alcanza top-2)
    Implies(Atom('CZE_1pt'), Atom('CZE_eliminado')),
    # Empate CZE–RSA ⇒ nadie ganó ese partido
    Implies(
        Atom('empate_CZE_RSA'),
        And(Not(Atom('CZE_vence_RSA')), Not(Atom('RSA_vence_CZE'))),
    ),
    Atom('empate_CZE_RSA'),
]

# ¿MEX tiene 9 pts y clasifica?
print(f"¿MEX tiene 9 pts? {check_entailment(kb_fifa, Atom('MEX_9pts'))}")
print(f"¿MEX clasifica? {check_entailment(kb_fifa, Atom('MEX_clasifica'))}")

# ¿CZE queda eliminado?
print(f"¿CZE eliminado? {check_entailment(kb_fifa, Atom('CZE_eliminado'))}")

# ¿CZE venció a RSA? (hubo empate → no)
print(f"¿CZE venció a RSA? {check_entailment(kb_fifa, Atom('CZE_vence_RSA'))}")
print(f"¿CZE NO venció a RSA? {check_entailment(kb_fifa, Not(Atom('CZE_vence_RSA')))}")


## Verificar tu implementación

Ejecuta las pruebas del Punto 1:

```bash
uv run pytest tests/test_model_checking.py -v
```

O la celda siguiente.


In [ ]:
!cd .. && uv run pytest tests/test_model_checking.py -v